# 01 · EDA — Exploración de datos

**Primer vistazo a los tres conjuntos crudos** del Hospital General de Medellín, tal como llegan de [datos.gov.co](https://www.datos.gov.co) (espejo local en `data/raw/`, refrescable con `uv run python -m src.data_pipeline.ingest`).

| Conjunto | Rol |
|---|---|
| `banco_sangre.csv` | Oferta: donaciones registradas |
| `atenciones.csv` | Demanda: población atendida/hospitalizada |
| `defunciones.csv` | Demanda: muertes con causa asociada |

Aquí solo se **observa** (tamaños, tipos, vacíos, rangos de fechas); la limpieza vive en [`02_limpieza_transformacion`](02_limpieza_transformacion.ipynb).

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_columns", 40)


In [2]:
RAW = ROOT / "data" / "raw"
crudos = {
    "banco_sangre": pd.read_csv(RAW / "banco_sangre.csv", low_memory=False),
    "atenciones": pd.read_csv(RAW / "atenciones.csv", low_memory=False),
    "defunciones": pd.read_csv(RAW / "defunciones.csv", low_memory=False),
}
pd.DataFrame(
    {nombre: {"filas": len(df), "columnas": df.shape[1]} for nombre, df in crudos.items()}
).T

,filas,columnas
banco_sangre,35840,11
atenciones,221203,12
defunciones,5094,26


## Muestra del banco de sangre

In [3]:
crudos["banco_sangre"].head(3)

,ANO,TRIMESTRE,FECHA EXTRACCION,RH,BARRIO,CIUDAD,EDAD,ESTATURA,FECHA NACIMIENTO,PESO,SEXO
0,2020,4,28/12/2020,0+,POPULAR 1,MEDELLIN,41.0,NaN,1982 Jan 01 12:00:00 AM,NaN,M
1,2020,1,01/02/2020,0+,20 DE JULIO,MEDELLIN,43.0,1.74,1979 Mar 12 12:00:00 AM,80.0,M
2,2020,1,01/02/2020,0+,20 DE JULIO,MEDELLIN,43.0,1.74,1979 Mar 12 12:00:00 AM,80.0,M


## Tipos de dato y datos faltantes por conjunto

In [4]:
resumen = []
for nombre, df in crudos.items():
    faltantes = df.isna().mean()
    resumen.append(
        {
            "conjunto": nombre,
            "columnas_numericas": int((df.dtypes != object).sum()),
            "columnas_texto": int((df.dtypes == object).sum()),
            "columnas_con_vacios": int((faltantes > 0).sum()),
            "peor_columna_%vacios": round(float(faltantes.max()) * 100, 1),
        }
    )
pd.DataFrame(resumen).set_index("conjunto")

,columnas_numericas,columnas_texto,columnas_con_vacios,peor_columna_%vacios
conjunto,,,,
banco_sangre,11,0,7,21.7
atenciones,12,0,5,26.2
defunciones,26,0,21,99.3


## Rango temporal de cada conjunto

In [5]:
rangos = []
for nombre, df in crudos.items():
    for col in df.columns:
        if "fecha" not in col.lower():
            continue
        fechas = pd.to_datetime(df[col], errors="coerce", format="mixed")
        if fechas.notna().mean() < 0.5:
            continue
        rangos.append(
            {
                "conjunto": nombre,
                "columna": col,
                "desde": fechas.min().date(),
                "hasta": fechas.max().date(),
                "registros_fechados": int(fechas.notna().sum()),
            }
        )
pd.DataFrame(rangos)

,conjunto,columna,desde,hasta,registros_fechados
0,banco_sangre,FECHA EXTRACCION,2020-01-02,2025-12-06,35840
1,banco_sangre,FECHA NACIMIENTO,1919-08-29,2023-01-20,35838
2,atenciones,Fecha atencion,2022-01-02,2026-12-03,221203
3,defunciones,FECHA DEFUNCION,2022-01-01,2026-12-03,5094


## Qué sigue

Con este panorama, [`02_limpieza_transformacion`](02_limpieza_transformacion.ipynb) depura cada conjunto y construye las series diarias de `data/processed/`; [`03_analisis_descriptivo`](03_analisis_descriptivo.ipynb) hace el EDA a fondo (hallazgos: riesgo estructural del O−, estacionalidad de diciembre, segmentación por edad).